In [0]:
spark

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Spotify_ETL").getOrCreate()

In [0]:
songs_daily_df = spark.read.parquet("/Volumes/mycatalog/niki/project/songs_parquet-20260625T123240Z-3-001/songs_parquet")

In [0]:
songs_daily_df.count()

42869655

In [0]:
from pyspark.sql.functions import min, max, count, when, col

# Date Validation
date_quality_check = (
    songs_daily_df
    .select(
        min("date").alias("first_chart_date"),
        max("date").alias("latest_chart_date"),
        min("release_date").alias("oldest_release_date"),
        max("release_date").alias("latest_release_date"),
        count(
            when(col("release_date") > col("date"), True)
        ).alias("future_release_errors")
    )
)

date_quality_check.show(truncate=False)

+----------------+-----------------+-------------------+-------------------+---------------------+
|first_chart_date|latest_chart_date|oldest_release_date|latest_release_date|future_release_errors|
+----------------+-----------------+-------------------+-------------------+---------------------+
|2017-01-01      |2026-05-28       |1840-06-14         |2026-05-31         |2659875              |
+----------------+-----------------+-------------------+-------------------+---------------------+



In [0]:
from pyspark.sql.functions import col

songs_daily_df.filter(
    col("release_date") > col("date")
).select(
    "track_name",
    "artist_names",
    "release_date",
    "date",
    "country",
    "rank"
).show(truncate=False)

+-------------+------------+------------+----------+-------+----+
|track_name   |artist_names|release_date|date      |country|rank|
+-------------+------------+------------+----------+-------+----+
|Azizam       |Ed Sheeran  |2025-09-12  |2025-04-04|cz     |167 |
|Azizam       |Ed Sheeran  |2025-09-12  |2025-04-07|cz     |121 |
|Azizam       |Ed Sheeran  |2025-09-12  |2025-04-08|cz     |121 |
|Azizam       |Ed Sheeran  |2025-09-12  |2025-04-09|cz     |66  |
|Azizam       |Ed Sheeran  |2025-09-12  |2025-04-10|cz     |63  |
|Azizam       |Ed Sheeran  |2025-09-12  |2025-04-11|cz     |66  |
|Azizam       |Ed Sheeran  |2025-09-12  |2025-04-12|cz     |109 |
|Azizam       |Ed Sheeran  |2025-09-12  |2025-04-13|cz     |126 |
|Azizam       |Ed Sheeran  |2025-09-12  |2025-04-14|cz     |128 |
|Azizam       |Ed Sheeran  |2025-09-12  |2025-04-15|cz     |129 |
|Azizam       |Ed Sheeran  |2025-09-12  |2025-04-16|cz     |138 |
|Azizam       |Ed Sheeran  |2025-09-12  |2025-04-17|cz     |163 |
|What Was 

In [0]:
invalid_records = songs_daily_df.filter(
    col("release_date") > col("date")
)

print(invalid_records.count())

2659875


In [0]:
len(songs_daily_df.columns)

18

In [0]:
songs_daily_df.printSchema()

root
 |-- date: string (nullable = true)
 |-- country: string (nullable = true)
 |-- rank: long (nullable = true)
 |-- uri: string (nullable = true)
 |-- artist_names: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- label: string (nullable = true)
 |-- peak_rank: long (nullable = true)
 |-- previous_rank: long (nullable = true)
 |-- days_on_chart: long (nullable = true)
 |-- streams: long (nullable = true)
 |-- consecutive_days: long (nullable = true)
 |-- entry_status: string (nullable = true)
 |-- peak_date: string (nullable = true)
 |-- entry_rank: long (nullable = true)
 |-- entry_date: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- artist_uris: string (nullable = true)



In [0]:
songs_daily_df.dtypes

[('date', 'string'),
 ('country', 'string'),
 ('rank', 'bigint'),
 ('uri', 'string'),
 ('artist_names', 'string'),
 ('track_name', 'string'),
 ('label', 'string'),
 ('peak_rank', 'bigint'),
 ('previous_rank', 'bigint'),
 ('days_on_chart', 'bigint'),
 ('streams', 'bigint'),
 ('consecutive_days', 'bigint'),
 ('entry_status', 'string'),
 ('peak_date', 'string'),
 ('entry_rank', 'bigint'),
 ('entry_date', 'string'),
 ('release_date', 'string'),
 ('artist_uris', 'string')]

In [0]:
from pyspark.sql.functions import countDistinct

for c in songs_daily_df.columns:
    print(c)
    songs_daily_df.select(
        countDistinct(c).alias("Unique Values")
    ).show()

date
+-------------+
|Unique Values|
+-------------+
|         3435|
+-------------+

country
+-------------+
|Unique Values|
+-------------+
|           73|
+-------------+

rank
+-------------+
|Unique Values|
+-------------+
|          200|
+-------------+

uri
+-------------+
|Unique Values|
+-------------+
|       242582|
+-------------+

artist_names
+-------------+
|Unique Values|
+-------------+
|        93331|
+-------------+

track_name
+-------------+
|Unique Values|
+-------------+
|       177356|
+-------------+

label
+-------------+
|Unique Values|
+-------------+
|        28028|
+-------------+

peak_rank
+-------------+
|Unique Values|
+-------------+
|          200|
+-------------+

previous_rank
+-------------+
|Unique Values|
+-------------+
|          201|
+-------------+

days_on_chart
+-------------+
|Unique Values|
+-------------+
|         3435|
+-------------+

streams
+-------------+
|Unique Values|
+-------------+
|      1235000|
+-------------+

consecutive

In [0]:
total_rows = songs_daily_df.count()

distinct_rows = songs_daily_df.distinct().count()

print("Total Rows :", total_rows)
print("Distinct Rows :", distinct_rows)
print("Duplicate Rows :", total_rows - distinct_rows)

Total Rows : 42869655
Distinct Rows : 42869655
Duplicate Rows : 0


## **Date**

In [0]:
songs_daily_df.select("date").dtypes

[('date', 'string')]

In [0]:
from pyspark.sql.functions import col

songs_daily_df.filter(col("date").isNull()).count()

0

In [0]:
from pyspark.sql.functions import col, to_date

invalid_date_count = songs_daily_df.filter(
    col("date").isNotNull() &
    to_date(col("date"), "yyyy-MM-dd").isNull()
).count()

print("Invalid Date Count:", invalid_date_count)

Invalid Date Count: 0


In [0]:
from pyspark.sql.functions import to_date, col

songs_daily_df = songs_daily_df.withColumn(
    "date",
    to_date(col("date"), "yyyy-MM-dd")
)

In [0]:
songs_daily_df.select("date").printSchema()

root
 |-- date: date (nullable = true)



In [0]:
from pyspark.sql.functions import min, max

songs_daily_df.select(
    min("date").alias("min_date"),
    max("date").alias("max_date")
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2017-01-01|2026-05-28|
+----------+----------+



In [0]:
songs_daily_df.select("date").distinct().orderBy("date").show(20, truncate=False)

+----------+
|date      |
+----------+
|2017-01-01|
|2017-01-02|
|2017-01-03|
|2017-01-04|
|2017-01-05|
|2017-01-06|
|2017-01-07|
|2017-01-08|
|2017-01-09|
|2017-01-10|
|2017-01-11|
|2017-01-12|
|2017-01-13|
|2017-01-14|
|2017-01-15|
|2017-01-16|
|2017-01-17|
|2017-01-18|
|2017-01-19|
|2017-01-20|
+----------+
only showing top 20 rows


## **Rank**

In [0]:
songs_daily_df.select("rank").dtypes

[('rank', 'bigint')]

In [0]:
from pyspark.sql.functions import col

songs_daily_df.filter(col("rank").isNull()).count()

0

In [0]:
songs_daily_df.select("rank").show(20)

+----+
|rank|
+----+
| 173|
| 174|
| 175|
| 176|
| 177|
| 178|
| 179|
| 180|
| 181|
| 182|
| 183|
| 184|
| 185|
| 186|
| 187|
| 188|
| 189|
| 190|
| 191|
| 192|
+----+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import min, max

songs_daily_df.select(
    min("rank").alias("Minimum Rank"),
    max("rank").alias("Maximum Rank")
).show()

+------------+------------+
|Minimum Rank|Maximum Rank|
+------------+------------+
|           1|         200|
+------------+------------+



In [0]:
from pyspark.sql.functions import col

invalid_rank_count = songs_daily_df.filter(
    (col("rank") < 1) | (col("rank") > 200)
).count()

print("Invalid Rank Count:", invalid_rank_count)

Invalid Rank Count: 0


## **Peak Rank**

In [0]:
#Step 1: Check Data Type
songs_daily_df.select("peak_rank").dtypes

[('peak_rank', 'bigint')]

In [0]:
#Step 2: Check Null Values
from pyspark.sql.functions import col

songs_daily_df.filter(col("peak_rank").isNull()).count()

0

In [0]:
#Step 3: Print Sample Values
songs_daily_df.select("peak_rank").show(20)

+---------+
|peak_rank|
+---------+
|        1|
|      108|
|      109|
|      122|
|       30|
|        2|
|      136|
|        1|
|       86|
|        7|
|       35|
|       19|
|        6|
|       59|
|        2|
|       18|
|       77|
|      118|
|        7|
|        1|
+---------+
only showing top 20 rows


In [0]:
#Step 4: Check Minimum and Maximum Value
from pyspark.sql.functions import min, max

songs_daily_df.select(
    min("peak_rank").alias("Minimum Peak Rank"),
    max("peak_rank").alias("Maximum Peak Rank")
).show()

+-----------------+-----------------+
|Minimum Peak Rank|Maximum Peak Rank|
+-----------------+-----------------+
|                1|              200|
+-----------------+-----------------+



In [0]:
#Step 5: Validate Business Range
from pyspark.sql.functions import col

invalid_peak_rank = songs_daily_df.filter(
    (col("peak_rank") < 1) | (col("peak_rank") > 200)
).count()

print("Invalid Peak Rank Count:", invalid_peak_rank)

Invalid Peak Rank Count: 0


In [0]:
#Step 5: Validate Business Range
songs_daily_df.select("peak_rank") \
              .distinct() \
              .orderBy("peak_rank") \
              .show(200)

+---------+
|peak_rank|
+---------+
|        1|
|        2|
|        3|
|        4|
|        5|
|        6|
|        7|
|        8|
|        9|
|       10|
|       11|
|       12|
|       13|
|       14|
|       15|
|       16|
|       17|
|       18|
|       19|
|       20|
|       21|
|       22|
|       23|
|       24|
|       25|
|       26|
|       27|
|       28|
|       29|
|       30|
|       31|
|       32|
|       33|
|       34|
|       35|
|       36|
|       37|
|       38|
|       39|
|       40|
|       41|
|       42|
|       43|
|       44|
|       45|
|       46|
|       47|
|       48|
|       49|
|       50|
|       51|
|       52|
|       53|
|       54|
|       55|
|       56|
|       57|
|       58|
|       59|
|       60|
|       61|
|       62|
|       63|
|       64|
|       65|
|       66|
|       67|
|       68|
|       69|
|       70|
|       71|
|       72|
|       73|
|       74|
|       75|
|       76|
|       77|
|       78|
|       79|
|       80|
|   

## **Previous Rank**

In [0]:
#Step 1: Check Data Type
songs_daily_df.select("previous_rank").dtypes

[('previous_rank', 'bigint')]

In [0]:
#Step 2: Check Null Values
from pyspark.sql.functions import col

songs_daily_df.filter(col("previous_rank").isNull()).count()

0

In [0]:
#Step 3: Print Sample Values
songs_daily_df.select("previous_rank").show(20)

+-------------+
|previous_rank|
+-------------+
|          161|
|          171|
|          109|
|          190|
|           -1|
|           -1|
|          144|
|          179|
|          198|
|          153|
|          183|
|          200|
|          194|
|           -1|
|          186|
|          174|
|          157|
|           -1|
|           -1|
|           -1|
+-------------+
only showing top 20 rows


In [0]:
#Step 4: Check Minimum and Maximum
from pyspark.sql.functions import min, max

songs_daily_df.select(
    min("previous_rank").alias("Minimum Previous Rank"),
    max("previous_rank").alias("Maximum Previous Rank")
).show()

+---------------------+---------------------+
|Minimum Previous Rank|Maximum Previous Rank|
+---------------------+---------------------+
|                   -1|                  200|
+---------------------+---------------------+



In [0]:
from pyspark.sql.functions import when, col

songs_daily_df = songs_daily_df.withColumn(
    "previous_rank",
    when(col("previous_rank") == -1, None)
    .otherwise(col("previous_rank"))
)

In [0]:
songs_daily_df.select(
    min("previous_rank").alias("Minimum Previous Rank"),
    max("previous_rank").alias("Maximum Previous Rank")
).show()

+---------------------+---------------------+
|Minimum Previous Rank|Maximum Previous Rank|
+---------------------+---------------------+
|                    1|                  200|
+---------------------+---------------------+



In [0]:
from pyspark.sql.functions import col

null_count = songs_daily_df.filter(
    col("previous_rank").isNull()
).count()

print("Null Count:", null_count)

Null Count: 2750762


In [0]:
from pyspark.sql.functions import col

songs_daily_df.filter(
    col("previous_rank").isNull()
).select(
    "track_name",
    "artist_names",
    "release_date",
    "date",
    "rank"
).show(20, truncate=False)

+--------------------------------+-----------------------------+------------+----------+----+
|track_name                      |artist_names                 |release_date|date      |rank|
+--------------------------------+-----------------------------+------------+----------+----+
|I Gotta Feeling                 |Black Eyed Peas              |2009-01-01  |2025-02-22|177 |
|Believer                        |Imagine Dragons              |2017-06-23  |2025-02-22|178 |
|Srouby a matice                 |Mandrage                     |2011-01-01  |2025-02-22|186 |
|Starships                       |Nicki Minaj                  |2012-01-01  |2025-02-22|190 |
|The Nights                      |Avicii                       |2014-01-01  |2025-02-22|191 |
|Shape of You                    |Ed Sheeran                   |2017-03-03  |2025-02-22|192 |
|Praha den a noc                 |CA$HANOVA BULHAR             |2022-12-24  |2025-02-22|195 |
|Titanium (feat. Sia)            |David Guetta|Sia          

## **Entry Rank**

In [0]:
#Step 1: Check Data Type
songs_daily_df.select("entry_rank").dtypes

[('entry_rank', 'bigint')]

In [0]:
#Step 2: Check Null Values
from pyspark.sql.functions import col

songs_daily_df.filter(
    col("entry_rank").isNull()
).count()

0

In [0]:
#Check Minimum and Maximum
from pyspark.sql.functions import min, max

songs_daily_df.select(
    min("entry_rank").alias("Minimum Entry Rank"),
    max("entry_rank").alias("Maximum Entry Rank")
).show()

+------------------+------------------+
|Minimum Entry Rank|Maximum Entry Rank|
+------------------+------------------+
|                 1|               200|
+------------------+------------------+



In [0]:
#Print Sample Values
songs_daily_df.select("entry_rank").show(20)

+----------+
|entry_rank|
+----------+
|         1|
|       166|
|       109|
|       150|
|        58|
|        73|
|       148|
|        20|
|       200|
|         7|
|       193|
|       174|
|       184|
|       191|
|        12|
|       156|
|       198|
|       118|
|        62|
|         1|
+----------+
only showing top 20 rows


In [0]:
#Validate Business Rule
from pyspark.sql.functions import col

invalid_entry_rank = songs_daily_df.filter(
    (col("entry_rank") < 1) |
    (col("entry_rank") > 200)
).count()

print("Invalid Entry Rank Count:", invalid_entry_rank)

Invalid Entry Rank Count: 0


In [0]:
#Check Distinct Values
songs_daily_df.select("entry_rank") \
              .distinct() \
              .orderBy("entry_rank") \
              .show(200)

+----------+
|entry_rank|
+----------+
|         1|
|         2|
|         3|
|         4|
|         5|
|         6|
|         7|
|         8|
|         9|
|        10|
|        11|
|        12|
|        13|
|        14|
|        15|
|        16|
|        17|
|        18|
|        19|
|        20|
|        21|
|        22|
|        23|
|        24|
|        25|
|        26|
|        27|
|        28|
|        29|
|        30|
|        31|
|        32|
|        33|
|        34|
|        35|
|        36|
|        37|
|        38|
|        39|
|        40|
|        41|
|        42|
|        43|
|        44|
|        45|
|        46|
|        47|
|        48|
|        49|
|        50|
|        51|
|        52|
|        53|
|        54|
|        55|
|        56|
|        57|
|        58|
|        59|
|        60|
|        61|
|        62|
|        63|
|        64|
|        65|
|        66|
|        67|
|        68|
|        69|
|        70|
|        71|
|        72|
|        73|
|        74|

## **Peak Date**

In [0]:
# Check Data Type
songs_daily_df.select("peak_date").dtypes

[('peak_date', 'string')]

In [0]:
# Check Null Values
from pyspark.sql.functions import col

songs_daily_df.filter(
    col("peak_date").isNull()
).count()

0

In [0]:
#Print Sample Values
songs_daily_df.select("peak_date").show(20, truncate=False)

+----------+
|peak_date |
+----------+
|2024-05-17|
|2025-02-10|
|2025-02-21|
|2025-01-20|
|2022-12-31|
|2017-07-20|
|2025-02-20|
|2021-10-22|
|2021-08-28|
|2024-11-08|
|2024-12-31|
|2024-06-29|
|2021-09-29|
|2024-08-17|
|2024-03-31|
|2022-03-06|
|2024-11-07|
|2023-12-31|
|2018-04-21|
|2017-01-06|
+----------+
only showing top 20 rows


In [0]:
#Validate Date Format
from pyspark.sql.functions import to_date, col

invalid_peak_date = songs_daily_df.filter(
    col("peak_date").isNotNull() &
    to_date(col("peak_date"), "yyyy-MM-dd").isNull()
).count()

print("Invalid Peak Date Count:", invalid_peak_date)

Invalid Peak Date Count: 0


In [0]:
#Check Date Range
from pyspark.sql.functions import min, max

songs_daily_df.select(
    min("peak_date").alias("Minimum Peak Date"),
    max("peak_date").alias("Maximum Peak Date")
).show()

+-----------------+-----------------+
|Minimum Peak Date|Maximum Peak Date|
+-----------------+-----------------+
|       2017-01-01|       2026-05-28|
+-----------------+-----------------+



In [0]:
# Business Validation
from pyspark.sql.functions import col

invalid_peak_date = songs_daily_df.filter(
    col("peak_date") < col("entry_date")
).count()

print("Peak Date before Entry Date:", invalid_peak_date)

Peak Date before Entry Date: 0


In [0]:
# Convert to DateType
from pyspark.sql.functions import to_date

songs_daily_df = songs_daily_df.withColumn(
    "peak_date",
    to_date("peak_date", "yyyy-MM-dd")
)

## **Entry Date**

In [0]:
# Check Data Type
songs_daily_df.select("entry_date").dtypes

[('entry_date', 'string')]

In [0]:
# Check Null Values
from pyspark.sql.functions import col

songs_daily_df.filter(
    col("entry_date").isNull()
).count()

0

In [0]:
from pyspark.sql.functions import to_date, col

invalid_entry_date = songs_daily_df.filter(
    col("entry_date").isNotNull() &
    to_date(col("entry_date"), "yyyy-MM-dd").isNull()
).count()

print("Invalid Entry Date Count:", invalid_entry_date)

Invalid Entry Date Count: 0


In [0]:
from pyspark.sql.functions import min, max

songs_daily_df.select(
    min("entry_date").alias("Minimum Entry Date"),
    max("entry_date").alias("Maximum Entry Date")
).show()

+------------------+------------------+
|Minimum Entry Date|Maximum Entry Date|
+------------------+------------------+
|        2017-01-01|        2026-05-28|
+------------------+------------------+



In [0]:
from pyspark.sql.functions import col

invalid_entry_date = songs_daily_df.filter(
    col("entry_date") > col("date")
).count()

print("Entry Date after Chart Date:", invalid_entry_date)

Entry Date after Chart Date: 0


In [0]:
from pyspark.sql.functions import to_date

songs_daily_df = songs_daily_df.withColumn(
    "entry_date",
    to_date("entry_date", "yyyy-MM-dd")
)

In [0]:
from pyspark.sql.functions import col

invalid_release_entry = songs_daily_df.filter(
    col("entry_date") < col("release_date")
).count()

print("Entry Date before Release Date:", invalid_release_entry)

Entry Date before Release Date: 10874897


In [0]:
from pyspark.sql.functions import col, datediff

songs_daily_df.filter(
    col("entry_date") < col("release_date")
).select(
    "track_name",
    "release_date",
    "entry_date",
    datediff(col("release_date"), col("entry_date")).alias("Difference_Days"),
    "rank"
).show(20, truncate=False)

+-----------------------------+------------+----------+---------------+----+
|track_name                   |release_date|entry_date|Difference_Days|rank|
+-----------------------------+------------+----------+---------------+----+
|Believer                     |2017-06-23  |2017-02-01|142            |178 |
|Praha/Vídeň                  |2022-03-11  |2021-09-17|175            |180 |
|Where Are You Now            |2023-11-10  |2021-09-01|800            |185 |
|Směj se teď                  |2024-10-31  |2024-03-22|223            |187 |
|Shape of You                 |2017-03-03  |2017-01-06|56             |192 |
|Poslední                     |2017-08-16  |2017-07-07|40             |193 |
|Praha den a noc              |2022-12-24  |2022-12-23|1              |195 |
|KSN                          |2024-11-22  |2024-11-21|1              |2   |
|APT.                         |2024-12-06  |2024-10-18|49             |6   |
|omluva                       |2025-01-17  |2024-12-28|20             |12  |

## **Release Date**

In [0]:
songs_daily_df.select("release_date").dtypes

[('release_date', 'string')]

In [0]:
# Check Null Values
from pyspark.sql.functions import col

songs_daily_df.filter(
    col("release_date").isNull()
).count()

638418

## **days_on_chart**

In [0]:
songs_daily_df.select("days_on_chart").dtypes

[('days_on_chart', 'bigint')]

In [0]:
from pyspark.sql.functions import col

songs_daily_df.filter(
    col("days_on_chart").isNull()
).count()

0

In [0]:
from pyspark.sql.functions import min, max

songs_daily_df.select(
    min("days_on_chart").alias("Minimum Days"),
    max("days_on_chart").alias("Maximum Days")
).show()

+------------+------------+
|Minimum Days|Maximum Days|
+------------+------------+
|           1|        3435|
+------------+------------+



In [0]:
from pyspark.sql.functions import col

invalid_days = songs_daily_df.filter(
    col("days_on_chart") <= 0
).count()

print("Invalid Days on Chart:", invalid_days)

Invalid Days on Chart: 0


In [0]:
# Print the Longest-Charting Songs
songs_daily_df.select(
    "date",
    "country",
    "track_name",
    "artist_names",
    "days_on_chart",
    "rank"
).orderBy(
    col("days_on_chart").desc()
).show(20, truncate=False)

+----------+-------+-------------------------------------------+-------------------------------------------------+-------------+----+
|date      |country|track_name                                 |artist_names                                     |days_on_chart|rank|
+----------+-------+-------------------------------------------+-------------------------------------------------+-------------+----+
|2026-05-28|sg     |你，好不好？ - TVBS連續劇【遺憾拼圖】片尾曲|Eric Chou                                        |3435         |91  |
|2026-05-28|ec     |Me Rehúso                                  |Danny Ocean                                      |3435         |8   |
|2026-05-28|bo     |Me Rehúso                                  |Danny Ocean                                      |3435         |8   |
|2026-05-28|id     |Untuk Perempuan Yang Sedang Di Pelukan     |Payung Teduh                                     |3434         |78  |
|2026-05-28|cr     |Me Rehúso                                  |Danny Ocean     

## **Consecutive Days**

In [0]:
songs_daily_df.select("consecutive_days").dtypes

[('consecutive_days', 'bigint')]

In [0]:
from pyspark.sql.functions import col

songs_daily_df.filter(
    col("consecutive_days").isNull()
).count()

0

In [0]:
from pyspark.sql.functions import min, max

songs_daily_df.select(
    min("consecutive_days").alias("Minimum Consecutive Days"),
    max("consecutive_days").alias("Maximum Consecutive Days")
).show()

+------------------------+------------------------+
|Minimum Consecutive Days|Maximum Consecutive Days|
+------------------------+------------------------+
|                       1|                    3435|
+------------------------+------------------------+



In [0]:
songs_daily_df.select("consecutive_days").show(20)

+----------------+
|consecutive_days|
+----------------+
|              20|
|              19|
|               2|
|              53|
|               1|
|               1|
|               6|
|              10|
|               3|
|              60|
|               3|
|               3|
|               6|
|               1|
|               2|
|              22|
|               4|
|               1|
|               1|
|               1|
+----------------+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import col

invalid_consecutive_days = songs_daily_df.filter(
    col("consecutive_days") <= 0
).count()

print("Invalid Consecutive Days:", invalid_consecutive_days)

Invalid Consecutive Days: 0


In [0]:
from pyspark.sql.functions import col

invalid_consecutive = songs_daily_df.filter(
    col("consecutive_days") > col("days_on_chart")
).count()

print("Consecutive Days > Days on Chart:", invalid_consecutive)

Consecutive Days > Days on Chart: 0


In [0]:
songs_daily_df.filter(
    col("consecutive_days") > col("days_on_chart")
).select(
    "track_name",
    "artist_names",
    "days_on_chart",
    "consecutive_days",
    "entry_date",
    "date"
).show(20, truncate=False)

+----------+------------+-------------+----------------+----------+----+
|track_name|artist_names|days_on_chart|consecutive_days|entry_date|date|
+----------+------------+-------------+----------------+----------+----+
+----------+------------+-------------+----------------+----------+----+



In [0]:
from pyspark.sql.functions import col

songs_daily_df.select(
    "track_name",
    "artist_names",
    "consecutive_days",
    "days_on_chart",
    "rank"
).orderBy(
    col("consecutive_days").desc()
).show(20, truncate=False)

+-------------------------------------------+------------+----------------+-------------+----+
|track_name                                 |artist_names|consecutive_days|days_on_chart|rank|
+-------------------------------------------+------------+----------------+-------------+----+
|你，好不好？ - TVBS連續劇【遺憾拼圖】片尾曲|Eric Chou   |3435            |3435         |91  |
|Me Rehúso                                  |Danny Ocean |3435            |3435         |8   |
|Me Rehúso                                  |Danny Ocean |3435            |3435         |8   |
|Me Rehúso                                  |Danny Ocean |3434            |3434         |6   |
|你，好不好？ - TVBS連續劇【遺憾拼圖】片尾曲|Eric Chou   |3434            |3434         |80  |
|Me Rehúso                                  |Danny Ocean |3434            |3434         |7   |
|Me Rehúso                                  |Danny Ocean |3433            |3433         |7   |
|你，好不好？ - TVBS連續劇【遺憾拼圖】片尾曲|Eric Chou   |3433            |3433         |62  |
|Me Rehús

## **Streams**

In [0]:
songs_daily_df.select("streams").dtypes

[('streams', 'bigint')]

In [0]:
from pyspark.sql.functions import col

songs_daily_df.filter(
    col("streams").isNull()
).count()

0

In [0]:
songs_daily_df.select("streams").show(20)

+-------+
|streams|
+-------+
|   9043|
|   8975|
|   8972|
|   8971|
|   8968|
|   8968|
|   8957|
|   8942|
|   8921|
|   8912|
|   8902|
|   8894|
|   8859|
|   8831|
|   8796|
|   8787|
|   8778|
|   8755|
|   8666|
|   8656|
+-------+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import min, max

songs_daily_df.select(
    min("streams").alias("Minimum Streams"),
    max("streams").alias("Maximum Streams")
).show()

+---------------+---------------+
|Minimum Streams|Maximum Streams|
+---------------+---------------+
|           1001|       30987370|
+---------------+---------------+



In [0]:
from pyspark.sql.functions import col

invalid_streams = songs_daily_df.filter(
    col("streams") <= 0
).count()

print("Invalid Streams:", invalid_streams)

Invalid Streams: 0


In [0]:
songs_daily_df.select("streams").summary().show()

+-------+-----------------+
|summary|          streams|
+-------+-----------------+
|  count|         42869655|
|   mean| 67841.5879753406|
| stddev|243998.0811227267|
|    min|             1001|
|    25%|             4781|
|    50%|            12190|
|    75%|            44789|
|    max|         30987370|
+-------+-----------------+



In [0]:
# Display Highest Streamed Songs
from pyspark.sql.functions import col

songs_daily_df.select(
    "track_name",
    "artist_names",
    "country",
    "date",
    "streams",
    "rank"
).orderBy(
    col("streams").desc()
).show(20, truncate=False)

+---------------------------------+------------------------+-------+----------+--------+----+
|track_name                       |artist_names            |country|date      |streams |rank|
+---------------------------------+------------------------+-------+----------+--------+----+
|The Fate of Ophelia              |Taylor Swift            |global |2025-10-03|30987370|1   |
|Fortnight (feat. Post Malone)    |Taylor Swift|Post Malone|global |2024-04-19|25204472|1   |
|All I Want for Christmas Is You  |Mariah Carey            |global |2024-12-24|24863570|1   |
|Last Christmas                   |Wham!                   |global |2024-12-24|24556791|2   |
|Elizabeth Taylor                 |Taylor Swift            |global |2025-10-03|23974787|2   |
|Opalite                          |Taylor Swift            |global |2025-10-03|23716087|3   |
|All I Want for Christmas Is You  |Mariah Carey            |global |2023-12-24|23701697|1   |
|All I Want for Christmas Is You  |Mariah Carey            |

## **Country**

In [0]:
songs_daily_df.select("country").dtypes

[('country', 'string')]

In [0]:
# Check Null Values
from pyspark.sql.functions import col

songs_daily_df.filter(
    col("country").isNull()
).count()

0

In [0]:
# Check Empty Strings
from pyspark.sql.functions import trim, col

songs_daily_df.filter(
    trim(col("country")) == ""
).count()

0

In [0]:
# Count Distinct Countries
songs_daily_df.select("country").distinct().count()

73

In [0]:
# Display All Country Codes
songs_daily_df.select("country") \
              .distinct() \
              .orderBy("country") \
              .show(100, truncate=False)

+-------+
|country|
+-------+
|ae     |
|ar     |
|at     |
|au     |
|be     |
|bg     |
|bo     |
|br     |
|by     |
|ca     |
|ch     |
|cl     |
|co     |
|cr     |
|cz     |
|de     |
|dk     |
|do     |
|ec     |
|ee     |
|eg     |
|es     |
|fi     |
|fr     |
|gb     |
|global |
|gr     |
|gt     |
|hk     |
|hn     |
|hu     |
|id     |
|ie     |
|il     |
|in     |
|is     |
|it     |
|jp     |
|kr     |
|kz     |
|lt     |
|lu     |
|lv     |
|ma     |
|mx     |
|my     |
|ng     |
|ni     |
|nl     |
|no     |
|nz     |
|pa     |
|pe     |
|ph     |
|pk     |
|pl     |
|pt     |
|py     |
|ro     |
|sa     |
|se     |
|sg     |
|sk     |
|sv     |
|th     |
|tr     |
|tw     |
|ua     |
|us     |
|uy     |
|ve     |
|vn     |
|za     |
+-------+



In [0]:
# Check Frequency of Each Country
from pyspark.sql.functions import desc

songs_daily_df.groupBy("country") \
              .count() \
              .orderBy(desc("count")) \
              .show(100, truncate=False)

+-------+------+
|country|count |
+-------+------+
|tw     |687000|
|it     |687000|
|se     |687000|
|br     |687000|
|no     |687000|
|pt     |687000|
|id     |687000|
|ca     |687000|
|au     |687000|
|nz     |687000|
|ec     |687000|
|pl     |687000|
|fr     |687000|
|fi     |687000|
|tr     |687000|
|be     |687000|
|my     |687000|
|mx     |687000|
|ph     |687000|
|pe     |687000|
|ar     |687000|
|dk     |686999|
|cr     |686999|
|co     |686999|
|ie     |686999|
|es     |686999|
|cl     |686998|
|gb     |686998|
|sg     |686998|
|hk     |686998|
|ch     |686997|
|us     |686997|
|nl     |686996|
|de     |686996|
|global |686995|
|at     |686993|
|cz     |685887|
|jp     |685399|
|gt     |681396|
|uy     |679667|
|hu     |672729|
|do     |670868|
|py     |665770|
|pa     |656252|
|th     |636857|
|bo     |635968|
|sv     |630440|
|hn     |614648|
|gr     |609731|
|sk     |595996|
|ro     |575737|
|vn     |572323|
|za     |559928|
|in     |529600|
|sa     |504123|
|lt     |48784

In [0]:
# Check Country Code Length
# Spotify uses 2-letter ISO country codes
from pyspark.sql.functions import length

songs_daily_df.filter(
    length(col("country")) != 2
).select("country").distinct().show(truncate=False)

+-------+
|country|
+-------+
|global |
+-------+



## **uri**

In [0]:
songs_daily_df.select("uri").dtypes

[('uri', 'string')]

In [0]:
# Check Null Values
from pyspark.sql.functions import col

songs_daily_df.filter(
    col("uri").isNull()
).count()

0

In [0]:
# Check Empty Strings
from pyspark.sql.functions import trim, col

songs_daily_df.filter(
    trim(col("uri")) == ""
).count()

0

In [0]:
# Count Distinct URIs
from pyspark.sql.functions import countDistinct

songs_daily_df.select(
    countDistinct("uri").alias("Unique URIs")
).show()

+-----------+
|Unique URIs|
+-----------+
|     242582|
+-----------+



In [0]:
# Check URI Format
from pyspark.sql.functions import col

invalid_uri = songs_daily_df.filter(
    ~col("uri").startswith("spotify:track:")
).count()

print("Invalid URI Format:", invalid_uri)

Invalid URI Format: 0


In [0]:
from pyspark.sql.functions import length

songs_daily_df.select(
    length("uri").alias("Length")
).groupBy("Length").count().orderBy("Length").show()

+------+--------+
|Length|   count|
+------+--------+
|    36|42869655|
+------+--------+



In [0]:
songs_daily_df.groupBy("uri").count().orderBy(col("count").desc()).show(20)

+--------------------+-----+
|                 uri|count|
+--------------------+-----+
|spotify:track:0Vj...|84216|
|spotify:track:7qi...|74910|
|spotify:track:7qE...|74062|
|spotify:track:0tg...|64402|
|spotify:track:7jt...|61972|
|spotify:track:0u2...|60351|
|spotify:track:2Qj...|54617|
|spotify:track:6Ue...|53359|
|spotify:track:4Dv...|52942|
|spotify:track:5uC...|51234|
|spotify:track:5Xe...|49711|
|spotify:track:2Vx...|49320|
|spotify:track:6RU...|47341|
|spotify:track:0fe...|43146|
|spotify:track:6dO...|42365|
|spotify:track:0pq...|41730|
|spotify:track:1JS...|41299|
|spotify:track:1rg...|40041|
|spotify:track:7BK...|39775|
|spotify:track:2Fx...|39379|
+--------------------+-----+
only showing top 20 rows


In [0]:
songs_daily_df.select("uri").show(20, truncate=False)

+------------------------------------+
|uri                                 |
+------------------------------------+
|spotify:track:7BRD7x5pt8Lqa1eGYC4dzj|
|spotify:track:1eTaznNW4Xxtx9za2SMTXB|
|spotify:track:6lKSOnC9iLGDonypMgvGrm|
|spotify:track:0d28khcov6AiegSCpG5TuT|
|spotify:track:4kLLWz7srcuLKA7Et40PQR|
|spotify:track:0pqnGHJpmpxLKifKRmU6WP|
|spotify:track:5ITV0zqzjOYfFWpW0xBmRa|
|spotify:track:163UXyyrFlAIQapVN3DqIp|
|spotify:track:0gucTLf7trAf37Ua1uAyAu|
|spotify:track:38pNZU3kgTMpP59E7fvJUh|
|spotify:track:0HPD5WQqrq7wPWR7P7Dw1i|
|spotify:track:09CnYHiZ5jGT1wr1TXJ9Zt|
|spotify:track:3mfER4ORePHvN35cbZ3dkV|
|spotify:track:3wfOyeMS8hHxxf8rzexktn|
|spotify:track:4DHktwgPm5VjDUPVNzD6Mp|
|spotify:track:2K7xn816oNHJZ0aVqdQsha|
|spotify:track:0hjzizkZxM9HCPGUQE7Xet|
|spotify:track:1oHNvJVbFkexQc0BpQp7Y4|
|spotify:track:0ct6r3EGTcMLPtrXHDvVjc|
|spotify:track:7qiZfU4dY1lWllzX7mPBI3|
+------------------------------------+
only showing top 20 rows


## **artist_uris**

In [0]:
songs_daily_df.select("artist_uris").dtypes

[('artist_uris', 'string')]

In [0]:
from pyspark.sql.functions import col

songs_daily_df.filter(
    col("artist_uris").isNull()
).count()

0

In [0]:
from pyspark.sql.functions import trim

songs_daily_df.filter(
    trim(col("artist_uris")) == ""
).count()

0

In [0]:
songs_daily_df.select("artist_uris").show(20, truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|artist_uris                                                                                                                                                                                                                        |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|spotify:artist:6qqNVTkY8uBg9cP3Jd7DAH                                                                                                                                                                                              |
|spotify:artist:4E2rKHVDssGJm2SCDOMMJB                                          

In [0]:
from pyspark.sql.functions import col

invalid_artist_uri = songs_daily_df.filter(
    ~col("artist_uris").contains("spotify:artist:")
).count()

print("Invalid Artist URI:", invalid_artist_uri)

Invalid Artist URI: 0


In [0]:
songs_daily_df.groupBy("artist_uris") \
              .count() \
              .orderBy(col("count").desc()) \
              .show(20, truncate=False)

+-------------------------------------+------+
|artist_uris                          |count |
+-------------------------------------+------+
|spotify:artist:4q3ewBCX7sLwd24euuV69X|524939|
|spotify:artist:6eUKZXaKkcviH0Ku9w2n3V|368958|
|spotify:artist:6qqNVTkY8uBg9cP3Jd7DAH|320675|
|spotify:artist:06HL4z0CvFAxyc27GXpf02|290442|
|spotify:artist:1Xyo4u8uXC1ZmMpatF05PJ|253065|
|spotify:artist:6KImCVD70vtIoJWnq6nGn3|209421|
|spotify:artist:66CXWjxzNUsdJxJ2JdwvnR|196522|
|spotify:artist:53XhwfbYqKCa1cC15pYq2q|187307|
|spotify:artist:3Nrfpe0tUJi4K4DXYWgMUX|184969|
|spotify:artist:6M2wZ9GZgrQXHCFfjv46we|184815|
|spotify:artist:15UsOTVnJzReFVN1VCnxy4|152382|
|spotify:artist:1McMsnEElThX1knmY4oliG|152001|
|spotify:artist:790FomKkXshlbRYZFtlgla|149367|
|spotify:artist:3TVXtAsR1Inumwj472S9r4|144268|
|spotify:artist:246dkjvS1zLTtiykXe5h60|137377|
|spotify:artist:4GNC7GD6oZMSxPGyXy4MNB|135746|
|spotify:artist:0Y5tJX1MQlPlqiwlOH1tJY|134048|
|spotify:artist:4gzpq5DPGxSnKTe4SA8HAU|133354|
|spotify:arti

## **Entry Status**

In [0]:
songs_daily_df.select("entry_status").dtypes

[('entry_status', 'string')]

In [0]:
from pyspark.sql.functions import col

songs_daily_df.filter(
    col("entry_status").isNull()
).count()

0

In [0]:
from pyspark.sql.functions import trim

songs_daily_df.filter(
    trim(col("entry_status")) == ""
).count()

0

In [0]:
songs_daily_df.select("entry_status") \
              .distinct() \
              .orderBy("entry_status") \
              .show(20, truncate=False)

+------------+
|entry_status|
+------------+
|MOVED_DOWN  |
|MOVED_UP    |
|NEW_ENTRY   |
|NO_CHANGE   |
|RE_ENTRY    |
+------------+



In [0]:
from pyspark.sql.functions import desc

songs_daily_df.groupBy("entry_status") \
              .count() \
              .orderBy(desc("count")) \
              .show(truncate=False)

+------------+--------+
|entry_status|count   |
+------------+--------+
|MOVED_DOWN  |18969613|
|MOVED_UP    |16904431|
|NO_CHANGE   |4244849 |
|RE_ENTRY    |2191236 |
|NEW_ENTRY   |559526  |
+------------+--------+



In [0]:
from pyspark.sql.functions import col

songs_daily_df.filter(
    col("entry_status") == "NEW"
).select(
    "track_name",
    "rank",
    "entry_rank",
    "days_on_chart"
).show(20, truncate=False)

+----------+----+----------+-------------+
|track_name|rank|entry_rank|days_on_chart|
+----------+----+----------+-------------+
+----------+----+----------+-------------+



In [0]:
songs_daily_df.filter(
    col("entry_status") == "SAME_POSITION"
).filter(
    col("rank") != col("previous_rank")
).count()

0

In [0]:
songs_daily_df.filter(
    col("entry_status") == "MOVE_UP"
).filter(
    col("rank") >= col("previous_rank")
).count()

0

In [0]:
songs_daily_df.filter(
    col("entry_status") == "MOVE_DOWN"
).filter(
    col("rank") <= col("previous_rank")
).count()

0

In [0]:
songs_daily_df.filter(
    col("entry_status") == "RE_ENTRY"
).select(
    "track_name",
    "days_on_chart",
    "consecutive_days",
    "previous_rank",
    "rank"
).show(20, truncate=False)

+--------------------------------+-------------+----------------+-------------+----+
|track_name                      |days_on_chart|consecutive_days|previous_rank|rank|
+--------------------------------+-------------+----------------+-------------+----+
|I Gotta Feeling                 |283          |1               |NULL         |177 |
|Believer                        |2719         |1               |NULL         |178 |
|Srouby a matice                 |300          |1               |NULL         |186 |
|Starships                       |19           |1               |NULL         |190 |
|The Nights                      |1245         |1               |NULL         |191 |
|Shape of You                    |1396         |1               |NULL         |192 |
|Praha den a noc                 |543          |1               |NULL         |195 |
|Titanium (feat. Sia)            |16           |1               |NULL         |196 |
|Summer                          |323          |1               |

## **Artist Name**

In [0]:
songs_daily_df.select("artist_names").dtypes

[('artist_names', 'string')]

In [0]:
from pyspark.sql.functions import trim, col

songs_daily_df.filter(
    trim(col("artist_names")) == ""
).count()

0

In [0]:
songs_daily_df.select("artist_names").show(20, truncate=False)

+-----------------------------------------------------------------+
|artist_names                                                     |
+-----------------------------------------------------------------+
|Billie Eilish                                                    |
|Doechii                                                          |
|Dove Cameron                                                     |
|Gorillaz|De La Soul                                              |
|Black Eyed Peas                                                  |
|Imagine Dragons                                                  |
|WizTheMc|bees & honey                                            |
|Calin                                                            |
|Nirvana                                                          |
|Viktor Sheen                                                     |
|Kesha                                                            |
|Dimitri Vegas & Like Mike|Tiësto|Dido|W&W|Dimit

In [0]:
from pyspark.sql.functions import col

songs_daily_df.filter(
    col("artist_names").isNull()
).count()

44711

In [0]:
from pyspark.sql.functions import col

songs_daily_df.filter(
    col("artist_names").isNull()
).select(
    "uri",
    "track_name",
    "artist_uris",
    "label",
    "country",
    "date"
).show(20, truncate=False)

+------------------------------------+----------+-------------------------------------+-----+-------+----------+
|uri                                 |track_name|artist_uris                          |label|country|date      |
+------------------------------------+----------+-------------------------------------+-----+-------+----------+
|spotify:track:3RXkboS74UYzN14xTqzPyY|NULL      |spotify:artist:0LyfQWJT6nXafLPZqxe9Of|NULL |de     |2017-07-20|
|spotify:track:4JAyIDXOqNM6qHuZML01uX|NULL      |spotify:artist:0LyfQWJT6nXafLPZqxe9Of|NULL |de     |2017-07-20|
|spotify:track:3bVbQvGVIe4n24AzyXovXh|NULL      |spotify:artist:0LyfQWJT6nXafLPZqxe9Of|NULL |de     |2017-07-20|
|spotify:track:3eFJqPe8VUYrABbFjSauuj|NULL      |spotify:artist:0LyfQWJT6nXafLPZqxe9Of|NULL |de     |2017-07-20|
|spotify:track:3RXkboS74UYzN14xTqzPyY|NULL      |spotify:artist:0LyfQWJT6nXafLPZqxe9Of|NULL |de     |2017-07-21|
|spotify:track:4JAyIDXOqNM6qHuZML01uX|NULL      |spotify:artist:0LyfQWJT6nXafLPZqxe9Of|NULL |de 

In [0]:
from pyspark.sql.functions import desc

songs_daily_df.filter(
    col("artist_names").isNull()
).groupBy("country") \
 .count() \
 .orderBy(desc("count")) \
 .show(20, truncate=False)

+-------+-----+
|country|count|
+-------+-----+
|in     |24585|
|pk     |7167 |
|vn     |2725 |
|se     |2645 |
|id     |1101 |
|de     |829  |
|py     |569  |
|cl     |561  |
|at     |524  |
|ch     |321  |
|no     |279  |
|th     |260  |
|br     |253  |
|ua     |228  |
|cr     |213  |
|ae     |171  |
|global |158  |
|pt     |157  |
|eg     |149  |
|fi     |132  |
+-------+-----+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import countDistinct, col

songs_daily_df.filter(
    col("artist_names").isNull()
).agg(
    countDistinct("uri").alias("Unique Songs with Missing Artist Name")
).show()

+-------------------------------------+
|Unique Songs with Missing Artist Name|
+-------------------------------------+
|                                  617|
+-------------------------------------+

